### 随机梯度下降  stochastic gradient descent

普通GD，也是批量梯度下降(Batch GD).

#### SGD

假设训练集的样本为$m$个，每次$w$下降用到的 $g=\frac{1}{m}(g_1+...+gm)$.

由于目标函数通常是包含样本平均，即$\frac{1}{n}$项，所以不需要直接对函数求梯度即可，不需要额外求一次平均。

但是一个epoch(理解为 完整使用训练集一次)，会更新 $m$ 次 $w$, 每次预先对m个样本随机打乱，一次仅使用一个样本 $g_i$, i为取到的样本i。

重复多个epoch,让参数有足够机会逐步接近最优解。

特点： 单步计算成本低，但是震荡大，不适合传统线搜索，一般使用固定/衰减学习率 $\alpha$.

#### 小批量梯度下降

与随机梯度下降，每次取batch_size个样本的梯度 平均，更新参数。




In [1]:
#import importlib
import model
#importlib.reload(model)

In [2]:
import numpy as np
from time import perf_counter

In [3]:
#随机梯度 或 小批量梯度 
def stochastic_descent(features,labels,batch_size,epochs=40,
                       initial_alpha=0.15,decay=0.08,seed=2026):
    rng=np.random.default_rng(seed)
    weights=np.zeros(features.shape[1])
    n_samples=len(labels)
    start_time=perf_counter()
    iters=0 #次数
    for epoch in range(epochs):
        indices=rng.permutation(n_samples)
        alpha=initial_alpha/(1+decay*epoch) #按训练次数衰减
        for start_index in range(0,n_samples,batch_size):
            batch=indices[start_index:start_index+batch_size]
            weights-=alpha*model.gradient(weights,features[batch],labels[batch])
            iters+=1
    name="SGD" if batch_size==1 else f"Mini-batch({batch_size})"
    return model.make_record(name,weights,features,labels,iters,perf_counter()-start_time)



In [4]:
def test_sgd():
    x_train, y_train, x_test, y_test, metadata = model.load_data()
    results=stochastic_descent(x_train,y_train,1)
    accuracy, precision,recall, log_loss = model.classification_metrics(
            results["weights"], x_test, y_test)
    print(
        f"method: {results.get('method')}\n"
        f"accuracy: {accuracy:.4f}\n"
        f"precision: {precision:.4f}\n"
        f"recall: {recall:.4f}\n"
        f"log_loss: {log_loss:.4f}"
    )

test_sgd()

method: SGD
accuracy: 0.9776
precision: 0.9553
recall: 0.8968
log_loss: 0.0571
